In [1]:
# ============================================================
# NOTEBOOK 1: DATASET PREPARATION
# Run this once — saves all datasets to /kaggle/working/datasets/
# ============================================================

import os
import json
import random
import pandas as pd
import numpy as np
import requests
from datasets import load_dataset

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Directory setup
base_path = '/kaggle/working/datasets'
os.makedirs(base_path, exist_ok=True)
os.makedirs(f'{base_path}/alpaca', exist_ok=True)
os.makedirs(f'{base_path}/oa', exist_ok=True)

print("="*60)
print("DATASET PREPARATION")
print("="*60)

# ============================================================
# 1. PILE DATASET (Streaming, 50k samples)
# ============================================================
print("\n[1/5] Loading PILE dataset (streaming)...")

pile_save_path = f'{base_path}/pile_50k.csv'

if os.path.exists(pile_save_path):
    print(f"   ✅ Already exists: {pile_save_path}")
    pile_df = pd.read_csv(pile_save_path)
    print(f"   Samples: {len(pile_df)}")
else:
    parts = [
        {'skip': 0,      'target': 17000, 'label': 'Start'},
        {'skip': 200000, 'target': 17000, 'label': 'Middle'},
        {'skip': 500000, 'target': 16000, 'label': 'End'},
    ]

    all_samples = []

    for part in parts:
        print(f"\n   📥 Collecting from {part['label']} (skip={part['skip']})...")

        dataset = load_dataset(
            "monology/pile-uncopyrighted",
            split="train",
            streaming=True
        )

        samples = []
        skipped = 0
        processed = 0

        for example in dataset:
            if skipped < part['skip']:
                skipped += 1
                continue

            text = example.get('text', '').strip()
            if len(text) > 100:
                samples.append({
                    'text': text,
                    'source': example.get('meta', {}).get('pile_set_name', 'unknown')
                })
                processed += 1

            if processed >= part['target']:
                break

            if processed % 5000 == 0 and processed > 0:
                print(f"      Collected {processed}/{part['target']}...")

        print(f"   ✅ Collected {len(samples)} from {part['label']}")
        all_samples.extend(samples)

    print(f"\n   🔀 Shuffling {len(all_samples)} samples...")
    random.shuffle(all_samples)

    pile_df = pd.DataFrame(all_samples[:50000])
    pile_df.to_csv(pile_save_path, index=False)
    print(f"   ✅ PILE saved: {len(pile_df)} samples → {pile_save_path}")
    print(f"   Source distribution:\n{pile_df['source'].value_counts().head(5)}")

# ============================================================
# 2. T-REx DATASET (KILT, streaming download)
# ============================================================
print("\n[2/5] Loading T-REx (KILT) dataset...")

trex_save_path = f'{base_path}/trex_50k.csv'

if os.path.exists(trex_save_path):
    print(f"   ✅ Already exists: {trex_save_path}")
    trex_df = pd.read_csv(trex_save_path)
    print(f"   Samples: {len(trex_df)}")
else:
    url = "http://dl.fbaipublicfiles.com/KILT/trex-train-kilt.jsonl"
    print(f"   Downloading from: {url}")
    response = requests.get(url, stream=True)

    samples = []
    target = 50000
    raw_lines = 0

    for line in response.iter_lines():
        if len(samples) >= target:
            break
        if not line:
            continue

        raw_lines += 1
        try:
            item = json.loads(line)
            input_text = item.get('input', '').strip()
            outputs = item.get('output', [])
            if not outputs:
                continue
            answer = outputs[0].get('answer', '').strip()
            if not answer:
                continue
            if '[SEP]' not in input_text:
                continue

            parts = input_text.split('[SEP]')
            subject = parts[0].strip()
            relation = parts[1].strip() if len(parts) > 1 else ''
            text = f"{subject}'s {relation} is"
            entity = answer

            samples.append({
                'text': text,
                'entity': entity,
                'subject': subject,
                'relation': relation
            })

        except Exception:
            continue

        if raw_lines % 100000 == 0:
            print(f"   Processed {raw_lines} lines, collected {len(samples)}...")

    response.close()
    trex_df = pd.DataFrame(samples[:target])
    trex_df.to_csv(trex_save_path, index=False)
    print(f"   ✅ T-REx saved: {len(trex_df)} samples → {trex_save_path}")

# ============================================================
# 3. MMLU DATASET
# ============================================================
print("\n[3/5] Loading MMLU dataset...")

mmlu_test_path = f'{base_path}/mmlu_test.csv'
mmlu_val_path  = f'{base_path}/mmlu_val.csv'

if os.path.exists(mmlu_test_path) and os.path.exists(mmlu_val_path):
    print(f"   ✅ Already exists")
    mmlu_df     = pd.read_csv(mmlu_test_path)
    mmlu_val_df = pd.read_csv(mmlu_val_path)
    print(f"   Test samples:  {len(mmlu_df)}")
    print(f"   Val samples:   {len(mmlu_val_df)}")
else:
    mmlu     = load_dataset("cais/mmlu", "all", split="test")
    mmlu_val = load_dataset("cais/mmlu", "all", split="validation")

    mmlu_df     = pd.DataFrame(mmlu)
    mmlu_val_df = pd.DataFrame(mmlu_val)

    mmlu_df.to_csv(mmlu_test_path, index=False)
    mmlu_val_df.to_csv(mmlu_val_path, index=False)
    print(f"   ✅ MMLU test saved:  {len(mmlu_df)} samples")
    print(f"   ✅ MMLU val saved:   {len(mmlu_val_df)} samples")
    print(f"   Subjects: {mmlu_df['subject'].nunique()}")

# ============================================================
# 4. ALPACA DATASET (Fine-tuning data)
# ============================================================
print("\n[4/5] Loading Alpaca dataset...")

alpaca_path = f'{base_path}/alpaca/alpaca_data.csv'

if os.path.exists(alpaca_path):
    print(f"   ✅ Already exists")
    alpaca_df = pd.read_csv(alpaca_path)
    print(f"   Samples: {len(alpaca_df)}")
else:
    alpaca_raw = load_dataset("tatsu-lab/alpaca", split="train")

    def format_alpaca(example):
        if example["input"] and example["input"].strip():
            text = (
                f"### Instruction:\n{example['instruction']}\n"
                f"### Input:\n{example['input']}\n"
                f"### Response:\n{example['output']}"
            )
        else:
            text = (
                f"### Instruction:\n{example['instruction']}\n"
                f"### Response:\n{example['output']}"
            )
        return {"text": text}

    alpaca_formatted = alpaca_raw.map(format_alpaca)
    alpaca_df = pd.DataFrame({
        'text': alpaca_formatted['text'],
        'instruction': alpaca_formatted['instruction'],
        'input': alpaca_formatted['input'],
        'output': alpaca_formatted['output'],
    })
    alpaca_df.to_csv(alpaca_path, index=False)
    print(f"   ✅ Alpaca saved: {len(alpaca_df)} samples → {alpaca_path}")

# ============================================================
# 5. OPENASSISTANT (OA) DATASET (Fine-tuning data)
# ============================================================
print("\n[5/5] Loading OpenAssistant (OA) dataset...")

oa_path = f'{base_path}/oa/oa_data.csv'

if os.path.exists(oa_path):
    print(f"   ✅ Already exists")
    oa_df = pd.read_csv(oa_path)
    print(f"   Samples: {len(oa_df)}")
else:
    oa_raw = load_dataset("OpenAssistant/oasst1", split="train")

    # Paper: extract single-turn English conversations
    oa_samples = []
    for example in oa_raw:
        if (example.get('lang') == 'en' and
            example.get('role') == 'prompter' and
            example.get('text')):

            text = (
                f"### Instruction:\n{example['text']}\n"
                f"### Response:\n"
            )
            oa_samples.append({
                'text': text,
                'raw_text': example['text'],
                'message_id': example.get('message_id', ''),
            })

    oa_df = pd.DataFrame(oa_samples)
    # Paper uses 11k pairs — sample down
    oa_df = oa_df.sample(n=min(11000, len(oa_df)), random_state=SEED).reset_index(drop=True)
    oa_df.to_csv(oa_path, index=False)
    print(f"   ✅ OA saved: {len(oa_df)} samples → {oa_path}")

# For fair comparison, sample Alpaca to same size as OA (paper does this)
alpaca_sampled_path = f'{base_path}/alpaca/alpaca_sampled.csv'
if not os.path.exists(alpaca_sampled_path):
    alpaca_sampled = alpaca_df.sample(
        n=min(11000, len(alpaca_df)),
        random_state=SEED
    ).reset_index(drop=True)
    alpaca_sampled.to_csv(alpaca_sampled_path, index=False)
    print(f"   ✅ Alpaca sampled to {len(alpaca_sampled)} (matching OA size)")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*60)
print("ALL DATASETS READY")
print("="*60)
print(f"  PILE       : {pile_save_path}")
print(f"  T-REx      : {trex_save_path}")
print(f"  MMLU test  : {mmlu_test_path}")
print(f"  MMLU val   : {mmlu_val_path}")
print(f"  Alpaca     : {alpaca_path}")
print(f"  Alpaca(11k): {alpaca_sampled_path}")
print(f"  OA         : {oa_path}")

DATASET PREPARATION

[1/5] Loading PILE dataset (streaming)...

   📥 Collecting from Start (skip=0)...


README.md:   0%|          | 0.00/776 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

      Collected 5000/17000...
      Collected 10000/17000...
      Collected 15000/17000...
   ✅ Collected 17000 from Start

   📥 Collecting from Middle (skip=200000)...


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

...

data/train-00000-of-00001-b42a775f407cee(…):   0%|          | 0.00/39.5M [00:00<?, ?B/s]

data/validation-00000-of-00001-134b8fd0c(…):   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

   ✅ OA saved: 11000 samples → /kaggle/working/datasets/oa/oa_data.csv
   ✅ Alpaca sampled to 11000 (matching OA size)

ALL DATASETS READY
  PILE       : /kaggle/working/datasets/pile_50k.csv
  T-REx      : /kaggle/working/datasets/trex_50k.csv
  MMLU test  : /kaggle/working/datasets/mmlu_test.csv
  MMLU val   : /kaggle/working/datasets/mmlu_val.csv
  Alpaca     : /kaggle/working/datasets/alpaca/alpaca_data.csv
  Alpaca(11k): /kaggle/working/datasets/alpaca/alpaca_sampled.csv
  OA         : /kaggle/working/datasets/oa/oa_data.csv


In [2]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 11.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 23.6 MB/s eta 0:00:0000:0100:01m


In [3]:
import os, gc, json, ast, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

SEED         = 42
EVAL_SAMPLES = 5000
N_BINS       = 10
EPOCHS       = 3

MODEL_PATH   = '/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1'
DATA_PATH    = '/kaggle/working/datasets'
RESULTS_PATH = '/kaggle/working/results'
CKPT_PATH    = '/kaggle/working/checkpoints'

os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(CKPT_PATH,    exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
print("All imports done.")

CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 | 15.6 GB
  GPU 1: Tesla T4 | 15.6 GB
All imports done.


In [4]:
def compute_ece(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(confidences)) * abs(
            accuracies[mask].mean() - confidences[mask].mean()
        )
    return float(ece)

def compute_reliability_diagram_data(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        bin_sizes.append(int(mask.sum()))
        if mask.sum() == 0:
            bin_accs.append(0.0)
            bin_confs.append(float((bins[i] + bins[i+1]) / 2))
        else:
            bin_accs.append(float(accuracies[mask].mean()))
            bin_confs.append(float(confidences[mask].mean()))
    return bin_accs, bin_confs, bin_sizes

def parse_choices(x):
    if isinstance(x, list): return x
    try: return ast.literal_eval(x)
    except: return x

print("ECE utilities ready.")

ECE utilities ready.


In [5]:


# 1. Load the raw files
pile_df     = pd.read_csv(f'{DATA_PATH}/pile_50k.csv')
trex_df     = pd.read_csv(f'{DATA_PATH}/trex_50k.csv')
mmlu_df     = pd.read_csv(f'{DATA_PATH}/mmlu_test.csv')
mmlu_val_df = pd.read_csv(f'{DATA_PATH}/mmlu_val.csv')

# 2. FIX MMLU TO 3000 SAMPLES (This is the new line to add)
mmlu_df = mmlu_df.sample(n=3000, random_state=SEED).reset_index(drop=True)

# 3. Process choices
mmlu_df['choices']     = mmlu_df['choices'].apply(parse_choices)
mmlu_val_df['choices'] = mmlu_val_df['choices'].apply(parse_choices)

# 4. FIX PILE AND TREX TO 5000 SAMPLES 
# (This uses the EVAL_SAMPLES = 5000 variable you set in Cell 3)
pile_eval = pile_df.sample(n=EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)
trex_eval = trex_df.sample(n=EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)

ANSWER_OPTIONS   = ['A', 'B', 'C', 'D']
PILE_PROMPT      = "Finish this sentence: "
TREX_PROMPT      = "Finish this Wikipedia description: "

print(f"  PILE eval  : {len(pile_eval)}") # Should show 5000
print(f"  T-REx eval : {len(trex_eval)}") # Should show 5000
print(f"  MMLU test  : {len(mmlu_df)}")    # Should show 3000
print(f"  MMLU val   : {len(mmlu_val_df)}")
print("Datasets ready.")

  PILE eval  : 5000
  T-REx eval : 5000
  MMLU test  : 3000
  MMLU val   : 1531
Datasets ready.


In [15]:
def evaluate_model(model, tokenizer, label="", include_prompt=True):
    """
    Runs all 3 tasks. include_prompt=True for alignment stage (paper Table 1).
    Returns dict with ECE + Accuracy for each task.
    """
    model.eval()
    results = {}

    pile_prompt  = PILE_PROMPT  if include_prompt else ""
    trex_prompt  = TREX_PROMPT  if include_prompt else ""

    # ── TASK 1: CLM ──────────────────────────────────────────
    print(f"\n  [{label}] CLM (PILE)...")
    confs, accs = [], []

    for idx, row in pile_eval.iterrows():
        text = pile_prompt + str(row['text']).strip()
        try:
            input_ids = tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=512
            )["input_ids"]
            seq_len = input_ids.shape[1]
            if seq_len < 3: continue

            pos        = np.random.randint(1, seq_len)
            context    = input_ids[:, :pos].to(model.device)
            true_token = input_ids[:, pos].item()

            with torch.no_grad():
                probs = F.softmax(
                    model(input_ids=context).logits[:, -1, :].float(),
                    dim=-1
                )
            confs.append(probs.max(dim=-1).values.item())
            accs.append(int(probs.argmax(dim=-1).item() == true_token))
        except: continue

        if (idx+1) % 10000 == 0:
            print(f"    {idx+1}/{EVAL_SAMPLES} | "
                  f"ECE: {compute_ece(confs, accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f}")

    results['CLM'] = {
        'ECE': compute_ece(confs, accs),
        'Accuracy': float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ CLM ECE: {results['CLM']['ECE']:.4f} | "
          f"Acc: {results['CLM']['Accuracy']:.4f}")

    # ── TASK 2: Facts ─────────────────────────────────────────
    print(f"\n  [{label}] Facts Generation (T-REx)...")
    confs, accs = [], []

    for idx, row in trex_eval.iterrows():
        text   = trex_prompt + str(row['text']).strip()
        entity = str(row['entity']).strip()
        try:
            entity_toks = tokenizer(
                entity, add_special_tokens=False, return_tensors="pt"
            )["input_ids"]
            if entity_toks.shape[1] == 0: continue
            true_first = entity_toks[0, 0].item()

            context_ids = tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=512
            )["input_ids"].to(model.device)

            with torch.no_grad():
                probs = F.softmax(
                    model(input_ids=context_ids).logits[:, -1, :].float(),
                    dim=-1
                )
            confs.append(probs.max(dim=-1).values.item())
            accs.append(int(probs.argmax(dim=-1).item() == true_first))
        except: continue

        if (idx+1) % 10000 == 0:
            print(f"    {idx+1}/{EVAL_SAMPLES} | "
                  f"ECE: {compute_ece(confs, accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f}")

    results['Facts'] = {
        'ECE': compute_ece(confs, accs),
        'Accuracy': float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ Facts ECE: {results['Facts']['ECE']:.4f} | "
          f"Acc: {results['Facts']['Accuracy']:.4f}")

    # ── TASK 3: MMLU ──────────────────────────────────────────
    print(f"\n  [{label}] MMLU...")
    confs, accs = [], []

    answer_token_ids = [
        tokenizer(opt, add_special_tokens=False,
                  return_tensors="pt")["input_ids"][0][0].item()
        for opt in ANSWER_OPTIONS
    ]

    def build_prompt(row):
        subject     = row['subject']
        prompt      = (f"The following are multiple choice questions "
                       f"(with answers) about {subject}.\n\n")
        subject_val = mmlu_val_df[mmlu_val_df['subject'] == subject]
        if len(subject_val) < 5:
            subject_val = mmlu_val_df
        shots = subject_val.sample(n=min(5, len(subject_val)),
                                   random_state=SEED)
        for _, shot in shots.iterrows():
            ch = shot['choices']
            prompt += f"Question: {shot['question']}\n"
            for i, opt in enumerate(ANSWER_OPTIONS):
                prompt += f"{opt}. {ch[i]}\n"
            prompt += f"Answer: {ANSWER_OPTIONS[int(shot['answer'])]}\n\n"
        ch = row['choices']
        prompt += f"Question: {row['question']}\n"
        for i, opt in enumerate(ANSWER_OPTIONS):
            prompt += f"{opt}. {ch[i]}\n"
        prompt += "Answer:"
        return prompt

    for idx, row in mmlu_df.iterrows():
        try:
            input_ids = tokenizer(
                build_prompt(row), return_tensors="pt",
                truncation=True, max_length=2048
            )["input_ids"].to(model.device)

            with torch.no_grad():
                logits = model(input_ids=input_ids).logits[:, -1, :]

            ans_probs = F.softmax(
                torch.tensor([logits[0, tid].item()
                              for tid in answer_token_ids]).float(),
                dim=-1
            )
            confs.append(ans_probs.max().item())
            accs.append(int(ans_probs.argmax().item() == int(row['answer'])))
        except: continue

        if (idx+1) % 2000 == 0:
            print(f"    {idx+1}/{len(mmlu_df)} | "
                  f"ECE: {compute_ece(confs, accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f}")

    results['MMLU'] = {
        'ECE': compute_ece(confs, accs),
        'Accuracy': float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ MMLU ECE: {results['MMLU']['ECE']:.4f} | "
          f"Acc: {results['MMLU']['Accuracy']:.4f}")

    return results

print("Evaluation function ready.")

Evaluation function ready.


In [16]:
print("Loading Alpaca dataset...")
alpaca_df = pd.read_csv(f'{DATA_PATH}/alpaca/alpaca_sampled.csv')
alpaca_dataset = Dataset.from_pandas(alpaca_df[['text']])
print(f"Alpaca samples: {len(alpaca_dataset)}")
print(f"Sample:\n{alpaca_df['text'].iloc[0][:200]}")

Loading Alpaca dataset...
Alpaca samples: 11000
Sample:
### Instruction:
Describe the character of the protagonist in the given TV show.
### Input:
Game of Thrones
### Response:
The protagonist of Game of Thrones is the longsuffering and honorable Eddard "


In [17]:
print("Loading LLaMA-7B...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Apply LoRA — matches paper Table 2 exactly
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model + LoRA ready.")

Loading LLaMA-7B...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622
Model + LoRA ready.


In [18]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=f'{CKPT_PATH}/alpaca_lora',
    num_train_epochs=1,                 # <--- CHANGED TO 1 EPOCH
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,      # effective batch = 4*8 = 32 per GPU
    learning_rate=3e-4,
    warmup_steps=100,
    lr_scheduler_type="linear",
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",              # save checkpoint after each epoch
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=2,
    
    # --- FIXED NAMING ---
    dataset_text_field="text",          
    max_length=2048,                
)

trainer = SFTTrainer(
    model=model,
    train_dataset=alpaca_dataset,
    args=training_args,
)

print("Starting Alpaca LoRA training...")
trainer.train()
print("Training complete.")

Adding EOS to train dataset:   0%|          | 0/11000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/11000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting Alpaca LoRA training...


Step,Training Loss
50,1.441239
100,1.111834
150,1.094083
200,1.087691
250,1.104387
300,1.089564


Training complete.


In [21]:
# Save the trained model immediately
save_path = f'{CKPT_PATH}/alpaca_lora_epoch1'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Model saved to: {save_path}")

# Verify files are there
import os
files = os.listdir(save_path)
print(f"Saved files: {files}")

✅ Model saved to: /kaggle/working/checkpoints/alpaca_lora_epoch1
Saved files: ['adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json', 'README.md', 'tokenizer_config.json']


In [26]:
from trl import SFTTrainer, SFTConfig

# Set to 2 epochs, because the model in memory already did 1
training_args = SFTConfig(
    output_dir=f'{CKPT_PATH}/alpaca_lora',
    num_train_epochs=2,                 # <--- 2 remaining epochs
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,      
    learning_rate=3e-4,
    warmup_steps=100,
    lr_scheduler_type="linear",
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",              
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=2,
    
    dataset_text_field="text",          
    max_length=2048,                
)

trainer = SFTTrainer(
    model=model,
    train_dataset=alpaca_dataset,
    args=training_args,
)

print("Continuing training for Epoch 2 and 3 using weights currently in memory...")

# Run without resume_from_checkpoint since the model is already updated in RAM
trainer.train()

print("Epoch 2 and 3 training complete.")

Adding EOS to train dataset:   0%|          | 0/11000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/11000 [00:00<?, ? examples/s]

Continuing training for Epoch 2 and 3 using weights currently in memory...


Step,Training Loss
50,1.085418
100,1.052107
150,1.047066
200,1.052955
250,1.081206
300,1.076269
350,1.084158
400,1.034686
450,1.029522
500,1.015852


Epoch 2 and 3 training complete.


In [27]:
import os

# Define a clear final directory for your trained adapter
final_save_path = f"{CKPT_PATH}/alpaca_lora_final_epochs_1to3"

# Create the directory if it doesn't exist
os.makedirs(final_save_path, exist_ok=True)

# 1. Save the trained model (LoRA adapter weights)
trainer.save_model(final_save_path)

# 2. Save the tokenizer so it matches your training setup
# (Assuming your tokenizer variable is named 'tokenizer')
tokenizer.save_pretrained(final_save_path)

print(f"✅ Final model and tokenizer successfully saved to: {final_save_path}")

# Optional: List the files saved to verify
print("Files saved:")
print(os.listdir(final_save_path))

✅ Final model and tokenizer successfully saved to: /kaggle/working/alpaca_lora_final_epochs_1to3
Files saved:
['adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json', 'README.md', 'training_args.bin', 'tokenizer_config.json']


In [34]:
!pip install -q bitsandbytes accelerate

In [39]:
import os
import json
import gc
import sys
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# =====================================================================
# 🧹 MEMORY WIPE (Clears previous corrupt states without restarting)
# =====================================================================
for var in ['base_model', 'eval_model', 'epoch_results']:
    if var in globals():
        del globals()[var]
if hasattr(sys, 'last_traceback'):
    del sys.last_traceback
gc.collect()
torch.cuda.empty_cache()
# =====================================================================

all_results = {"model": "LLaMA-7B", "method": "Alpaca_LoRA", "epochs": {}}
ckpt_base_dir = f'{CKPT_PATH}/alpaca_lora'

ckpt_folders = [d for d in os.listdir(ckpt_base_dir) if d.startswith('checkpoint')]
ckpt_folders.sort(key=lambda x: int(x.split('-')[-1]))

for epoch in range(1, len(ckpt_folders) + 1):
    print(f"{'='*60}")
    print(f"EVALUATING EPOCH {epoch}")
    print(f"{'='*60}")
    print(f"Available checkpoints: {ckpt_folders}")
    
    ckpt_dir = f'{ckpt_base_dir}/{ckpt_folders[epoch-1]}'

    # 1. Load a pristine, uncorrupted base model for EVERY epoch
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    
    # 2. Wrap the base model with the correct LoRA checkpoint using PeftModel
    eval_model = PeftModel.from_pretrained(base_model, ckpt_dir)
    eval_model.eval()

    # 3. Evaluate using your custom function
    epoch_results = evaluate_model(
        eval_model, tokenizer,
        label=f"Alpaca_LoRA_Epoch{epoch}",
        include_prompt=True
    )
    
    all_results["epochs"][f"epoch_{epoch}"] = epoch_results

    # Exact summary format requested
    print(f"\n  Epoch {epoch} Summary:")
    print(f"  CLM   ECE: {epoch_results['CLM']['ECE']:.4f} | Acc: {epoch_results['CLM']['Accuracy']:.4f}")
    print(f"  Facts ECE: {epoch_results['Facts']['ECE']:.4f} | Acc: {epoch_results['Facts']['Accuracy']:.4f}")
    print(f"  MMLU  ECE: {epoch_results['MMLU']['ECE']:.4f} | Acc: {epoch_results['MMLU']['Accuracy']:.4f}")

    os.makedirs(RESULTS_PATH, exist_ok=True)
    with open(f'{RESULTS_PATH}/alpaca_lora_results.json', 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"  ✅ Saved results so far.\n")

    # 4. CRITICAL: Ruthlessly delete the model to prevent OOM on the next loop
    del eval_model
    del base_model
    del epoch_results
    gc.collect()
    torch.cuda.empty_cache()

EVALUATING EPOCH 1
Available checkpoints: ['checkpoint-344', 'checkpoint-688']


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


  [Alpaca_LoRA_Epoch1] CLM (PILE)...
    ✅ CLM ECE: 0.0286 | Acc: 0.6018

  [Alpaca_LoRA_Epoch1] Facts Generation (T-REx)...
    ✅ Facts ECE: 0.2300 | Acc: 0.0314

  [Alpaca_LoRA_Epoch1] MMLU...
    2000/3000 | ECE: 0.0785 | Acc: 0.4360
    ✅ MMLU ECE: 0.0717 | Acc: 0.4450

  Epoch 1 Summary:
  CLM   ECE: 0.0286 | Acc: 0.6018
  Facts ECE: 0.2300 | Acc: 0.0314
  MMLU  ECE: 0.0717 | Acc: 0.4450
  ✅ Saved results so far.

EVALUATING EPOCH 2
Available checkpoints: ['checkpoint-344', 'checkpoint-688']


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


  [Alpaca_LoRA_Epoch2] CLM (PILE)...
    ✅ CLM ECE: 0.0322 | Acc: 0.6030

  [Alpaca_LoRA_Epoch2] Facts Generation (T-REx)...
    ✅ Facts ECE: 0.2327 | Acc: 0.0458

  [Alpaca_LoRA_Epoch2] MMLU...
    2000/3000 | ECE: 0.1015 | Acc: 0.4485
    ✅ MMLU ECE: 0.0970 | Acc: 0.4557

  Epoch 2 Summary:
  CLM   ECE: 0.0322 | Acc: 0.6030
  Facts ECE: 0.2327 | Acc: 0.0458
  MMLU  ECE: 0.0970 | Acc: 0.4557
  ✅ Saved results so far.



In [51]:
import os
import json
import torch
import gc
from transformers import AutoModelForCausalLM
from peft import PeftModel

# 1. Force clear any lingering memory
gc.collect()
torch.cuda.empty_cache()

epoch = 3
# Directly target your final saved checkpoint
ckpt_dir = f'{CKPT_PATH}/alpaca_lora/checkpoint-688'
results_file_path = os.path.join(RESULTS_PATH, 'alpaca_lora_results.json')

print(f"\nEvaluating Epoch {epoch}...")

# 2. Safely load previous results
if os.path.exists(results_file_path):
    with open(results_file_path, 'r') as f:
        all_results = json.load(f)
else:
    all_results = {"model": "LLaMA-7B", "method": "Alpaca_LoRA", "epochs": {}}

# 3. Load Base Model safely (low_cpu_mem_usage helps prevent RAM spikes)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True 
)

# Fix padding token to prevent math errors
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
base_model.config.pad_token_id = tokenizer.pad_token_id

# 4. Load your saved LoRA weights onto the base model
eval_model = PeftModel.from_pretrained(base_model, ckpt_dir)
eval_model.eval()

# 5. Evaluate (using torch.no_grad() to save GPU memory during inference)
with torch.no_grad():
    epoch_results = evaluate_model(
        eval_model, tokenizer,
        label=f"Alpaca_LoRA_Epoch{epoch}",
        include_prompt=True
    )

all_results["epochs"][f"epoch_{epoch}"] = epoch_results

# 6. Exact summary format requested
print(f"\n  Epoch {epoch} Summary:")
print(f"  CLM   ECE: {epoch_results['CLM']['ECE']:.4f} | Acc: {epoch_results['CLM']['Accuracy']:.4f}")
print(f"  Facts ECE: {epoch_results['Facts']['ECE']:.4f} | Acc: {epoch_results['Facts']['Accuracy']:.4f}")
print(f"  MMLU  ECE: {epoch_results['MMLU']['ECE']:.4f} | Acc: {epoch_results['MMLU']['Accuracy']:.4f}")

# 7. Save results
os.makedirs(RESULTS_PATH, exist_ok=True)
with open(results_file_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print("  ✅ Saved results so far.")

# Clean up immediately after finishing
del eval_model
del base_model
gc.collect()
torch.cuda.empty_cache()


Evaluating Epoch 3...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


  [Alpaca_LoRA_Epoch3] CLM (PILE)...
    ✅ CLM ECE: 0.0856 | Acc: 0.5079

  [Alpaca_LoRA_Epoch3] Facts Generation (T-REx)...
    ✅ Facts ECE: 0.2323 | Acc: 0.0462

  [Alpaca_LoRA_Epoch3] MMLU...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


    ✅ MMLU ECE: 0.0000 | Acc: nan

  Epoch 3 Summary:
  CLM   ECE: 0.0856 | Acc: 0.5079
  Facts ECE: 0.2323 | Acc: 0.0462
  MMLU  ECE: 0.0000 | Acc: nan
  ✅ Saved results so far.
